<a href="https://colab.research.google.com/github/J0SAL/genai-projects/blob/main/8-multi-agent-research/8_multi_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Research Agent with Open Models

In [1]:
%pip install -Uq "smolagents[mcp,litellm,openai]" huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.4/684.4 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 15.2 MB/s eta 0:00:00


In [11]:
import os
import getpass
from google.colab import userdata
# os.environ["FIRECRAWL_API_KEY"] = getpass.getpass("Firecrawl API Key:")
# os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face Token:")
# above added in colab env

In [22]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    api_key=userdata.get("HF_TOKEN"),
)

completion = client.chat.completions.create(
    model="deepseek-ai/DeepSeek-V4-Pro:fireworks-ai",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ],
)

print(completion.choices[0].message)

ChatCompletionOutputMessage(role='assistant', content='The capital of France is **Paris**.', reasoning=None, tool_call_id=None, tool_calls=None, reasoning_content='We are asked: "What is the capital of France?" This is a simple factual question. The answer is Paris. No need for further explanation. Just respond directly.')


## Generate research plan

First, we will have the LLM generate a research plan that will serve as anchor for all our sub agents.

In [25]:
PLANNER_SYSTEM_INSTRUCTIONS= """
You will be given a research task by a user. Your job is to produce a set of
instructions for a researcher that will complete the task. Do NOT complete the
task yourself, just provide instructions on how to complete it.

GUIDELINES:
1. Maximize specificity and detail. Include all known user preferences and
   explicitly list key attributes or dimensions to consider.
2. If essential attributes are missing, explicitly state that they are open-ended.
3. Avoid unwarranted assumptions. Treat unspecified dimensions as flexible.
4. Use the first person (from the user's perspective).
5. When helpful, explicitly ask the researcher to include tables.
6. Include the expected output format (e.g. structured report with headers).
7. Preserve the input language unless the user explicitly asks otherwise.
8. Sources: prefer primary / official / original sources.
"""


from huggingface_hub import InferenceClient

def generate_research_plan(user_query: str) -> str:
    MODEL_ID = "deepseek-ai/DeepSeek-V4-Pro:fireworks-ai"

    print("Generating the research plan for the query: ", user_query)
    print("MODEL with provider: ", MODEL_ID)
    planner_client = InferenceClient(
        api_key=userdata.get("HF_TOKEN")
    )
    completion = planner_client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": PLANNER_SYSTEM_INSTRUCTIONS},
            {"role": "user", "content": user_query},
        ],
    )

    research_plan = completion.choices[0].message.content

    print("\033[93mGenerated Research Plan\033[0m")
    print(f"\033[93m{research_plan}\033[0m")

    return research_plan

research_plan = generate_research_plan("research about the climate in india")

Generating the research plan for the query:  research about the climate in india
MODEL with provider:  deepseek-ai/DeepSeek-V4-Pro:fireworks-ai
Generated Research Plan
Based on your request to research India’s climate, I’ve prepared a detailed set of instructions for the researcher. As I don’t have additional preferences from you, I’ve assumed you want a thorough, well-sourced overview that covers the country’s climate characteristics, regional diversity, seasonal patterns, and emerging trends. The researcher should follow these instructions exactly, but they may ask for clarification if needed. Please note that any unspecified dimensions (e.g., time period, specific regions) should be treated flexibly and noted as open-ended in their work.

---

### Research Objective
Research India’s climate comprehensively. The researcher must produce a structured report that explains the dominant climate systems, seasonal cycles, regional variations, and current challenges related to climate change

# Split into subtasks
Next, we will split the research plan into multiple research subtasks that will be run by our subagents agents.


In [34]:
import json
from pydantic import BaseModel, Field
from typing import List
from pprint import pprint

TASK_SPLITTER_SYSTEM_INSTRUCTIONS = """
You will be given a set of research instructions (a research plan).
Your job is to break this plan into a set of coherent, non-overlapping
subtasks that can be researched independently by separate agents.

Requirements:
- 3 to 8 subtasks is usually a good range. Use your judgment.
- Each subtask should have:
  - an 'id' (short string),
  - a 'title' (short descriptive title),
  - a 'description' (clear, detailed instructions for the sub-agent).
- Subtasks should collectively cover the full scope of the original plan
  without unnecessary duplication.
- Prefer grouping by dimensions: time periods, regions, actors, themes,
  causal mechanisms, etc., depending on the topic.
- Each description should be very clear and detailed about everything that
  the agent needs to research to cover that topic.
- Do not include a final task that will put everything together.
  This will be done later in another step.

Output format:
Return ONLY valid JSON with this schema:

{
  "subtasks": [
    {
      "id": "string",
      "title": "string",
      "description": "string"
    }
  ]
}
"""

class Subtask(BaseModel):
    id: str = Field(
        ...,
        description="Short identifier for the subtask (e.g. 'A', 'history', 'drivers').",
    )
    title: str = Field(
        ...,
        description="Short descriptive title of the subtask.",
    )
    description: str = Field(
        ...,
        description="Clear, detailed instructions for the sub-agent that will research this subtask.",
    )

class SubtaskList(BaseModel):
    subtasks: List[Subtask] = Field(
        ...,
        description="List of subtasks that together cover the whole research plan.",
    )

TASK_SPLITTER_JSON_SCHEMA = {
    "name": "subtaskList",
    "schema": SubtaskList.model_json_schema(),
    "strict": True,
}


def split_into_subtasks(research_plan: str):

    MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507:fastest"

    print("Splitting the research plan into subtasks...")
    print("MODEL: ", MODEL_ID)

    client = InferenceClient(
      api_key=userdata.get("HF_TOKEN")
    )

    completion = client.chat_completion(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": TASK_SPLITTER_SYSTEM_INSTRUCTIONS},
            {"role": "user", "content": research_plan},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": TASK_SPLITTER_JSON_SCHEMA,
        },
    )

    message = completion.choices[0].message

    print(message)

    subtasks = json.loads(message.content)['subtasks']

    print("\033[93mGenerated The Following Subtasks\033[0m")
    for task in subtasks:
      print(f"\033[93m{task['title']}\033[0m")
      pprint(f"\033[93m{task['description']}\033[0m")
      print()

    return subtasks

subtasks = split_into_subtasks(research_plan)



Splitting the research plan into subtasks...
MODEL:  Qwen/Qwen3-4B-Instruct-2507:fastest
ChatCompletionOutputMessage(role='assistant', content='{\n  "subtasks": [\n    {\n      "description": "Conduct a comprehensive analysis of India\'s climatic zones using the Köppen-Geiger classification system. Identify all applicable climate types (e.g., tropical monsoon, humid subtropical, hot-temperate desert, montane) across the subcontinent. Describe their spatial distribution by referencing major geographical boundaries: Himalayas, Thar Desert, Western and Eastern Ghats, Indo-Gangetic Plain, and coastal regions. Include a detailed table summarizing climate types, representative states/regions, average temperatures, and precipitation patterns. Cross-reference with official IMD and MoES publications to validate classifications and identify any regional nuances or updates in Köppen classification for South Asia.",\n      "id": "koppengeiger",\n      "title": "Köppen-Geiger Climate Types and Spat

# Create subagents + coordinator

In this step, we’ll create a tool that spins up a dedicated sub-agent for each subtask. This tool will be handed to the Coordinator agent, which will invoke it whenever a new subtask needs to be processed. Each sub-agent will perform thorough research on its assigned subtask and return its findings once completed. The Coordinator will then aggregate all sub-agent outputs into a comprehensive deep-research report.

In [42]:
SUBAGENT_PROMPT_TEMPLATE = """
You are a specialized research sub-agent.

Global user query:
{user_query}

Overall research plan:
{research_plan}

Your specific subtask (ID: {subtask_id}, Title: {subtask_title}) is:

\"\"\"{subtask_description}\"\"\"

Instructions:
- Focus ONLY on this subtask, but keep the global query in mind for context.
- Use the available tools to search for up-to-date, high-quality sources.
- When calling firecrawl_search, include ALL required fields from the schema,
even if empty or default values must be used.
- Prioritize primary and official sources when possible.
- Be explicit about uncertainties, disagreements in the literature, and gaps.
- Return your results as a MARKDOWN report with this structure:

# [Subtask ID] [Subtask Title]

## Summary
Short overview of the main findings.

## Detailed Analysis
Well-structured explanation with subsections as needed.

## Key Points
- Bullet point
- Bullet point

## Sources
- [Title](url) - short comment on why this source is relevant

Now perform the research and return ONLY the markdown report.
"""

COORDINATOR_PROMPT_TEMPLATE = """
You are the LEAD RESEARCH COORDINATOR AGENT.

The user has asked:
\"\"\"{user_query}\"\"\"

A detailed research plan has already been created:

\"\"\"{research_plan}\"\"\"

This plan has been split into the following subtasks (JSON):

```json
{subtasks_json}
```
Each element has the shape:
{{
“id”: “timeframe_confirmation”,
“title”: “Confirm Research Scope Parameters”,
“description”: “Analyze the scope parameters…”
}}

You have access to a tool called:
• initialize_subagent(subtask_id: str, subtask_title: str, subtask_description: str)

Your job:
1. For EACH subtask in the JSON array, call initialize_subagent exactly once
with:
• subtask_id       = subtask[“id”]
• subtask_title    = subtask[“title”]
• subtask_description = subtask[“description”]
2. Wait for all sub-agent reports to come back. Each tool call returns a
markdown report for that subtask.
3. After you have results for ALL subtasks, synthesize them into a SINGLE,
coherent, deeply researched report addressing the original user query
("{user_query}").

Final report requirements:
• Integrate all sub-agent findings; avoid redundancy.
• Make the structure clear with headings and subheadings.
• Highlight:
• key drivers and mechanisms of insecurity,
• historical and temporal evolution,
• geographic and thematic patterns,
• state capacity, public perception, and socioeconomic correlates,
• open questions and uncertainties.
• Include final sections:
• Open Questions and Further Research
• Bibliography / Sources: merge and deduplicate the key sources from all sub-agents.

Important:
• DO NOT expose internal tool-call mechanics to the user.
• Your final answer to the user should be a polished markdown report.
"""


In [43]:
from smolagents import InferenceClientModel, MCPClient, tool, ToolCallingAgent
import os

FIRECRAWL_API_KEY = userdata.get("FIRECRAWL_API_KEY")
MCP_URL = f"https://mcp.firecrawl.dev/{FIRECRAWL_API_KEY}/v2/mcp"

COORDINATOR_MODEL_ID = "openai/gpt-oss-120b"
SUBAGENT_MODEL_ID = "openai/gpt-oss-120b"
PROVIDER = "groq"

def run_deep_research(user_query: str) -> str:
    print("Running the deep research...")

    # 1) Generate research plan
    research_plan = generate_research_plan(user_query)

    # 2) Split into explicit subtasks
    subtasks = split_into_subtasks(research_plan)

    print("Initializing Coordinator")
    print("Coordinator Model: ", COORDINATOR_MODEL_ID)
    print("Subagent Model: ", SUBAGENT_MODEL_ID)
    print("Provider: ", PROVIDER)

    coordinator_model = InferenceClientModel(
        model_id=COORDINATOR_MODEL_ID,
        provider=PROVIDER,
        token=userdata.get("HF_TOKEN")
    )

    subagent_model = InferenceClientModel(
        model_id=COORDINATOR_MODEL_ID,
        provider=PROVIDER,
        token=userdata.get("HF_TOKEN")
    )

    with MCPClient({"url": MCP_URL, "transport": "streamable-http"}) as mcp_tools:
        @tool
        def initialize_subagent(subtask_id: str, subtask_title: str, subtask_description: str) -> str:
            """
           Spawn a dedicated research sub-agent for a single subtask.

            Args:
                subtask_id (str): The unique identifier for the subtask.
                subtask_title (str): The descriptive title of the subtask.
                subtask_description (str): Detailed instructions for the sub-agent to perform the subtask.

            The sub-agent:
            - Has access to the Firecrawl MCP tools.
            - Must perform deep research ONLY on this subtask.
            - Returns a structured markdown report with:
              - a clear heading identifying the subtask,
              - a narrative explanation,
              - bullet-point key findings,
              - explicit citations / links to sources.
            """
            print(f"Initializing Subagent for task {subtask_id}...")

            subagent = ToolCallingAgent(
                tools=mcp_tools,                # Firecrawl MCP toolkit
                model=subagent_model,
                add_base_tools=False,
                name=f"subagent_{subtask_id}",
            )
            subagent_prompt = SUBAGENT_PROMPT_TEMPLATE.format(
                user_query=user_query,
                research_plan=research_plan,
                subtask_id=subtask_id,
                subtask_title=subtask_title,
                subtask_description=subtask_description,
            )

            return subagent.run(subagent_prompt)

        coordinator = ToolCallingAgent(
            tools=[initialize_subagent],
            model=coordinator_model,
            add_base_tools=False,
            name="coordinator_agent"
        )

        subtasks_json = json.dumps(subtasks, indent=2, ensure_ascii=False)

        coordinator_prompt = COORDINATOR_PROMPT_TEMPLATE.format(
            user_query=user_query,
            research_plan=research_plan,
            subtasks_json=subtasks_json,
        )

        final_report = coordinator.run(coordinator_prompt)

        return final_report

# Final result

This is the result of running the entire Deep Research Pipeline

In [44]:
result = run_deep_research("research the climate in india")

Running the deep research...
Generating the research plan for the query:  research the climate in india
MODEL with provider:  deepseek-ai/DeepSeek-V4-Pro:fireworks-ai
Generated Research Plan
I need a detailed, structured research report on the climate of India. Please follow the instructions below to gather, analyze, and present the information. Do not complete the research yourself; rather, use these instructions as the task specification.

**Objective**  
Produce a comprehensive analysis of India’s climate, covering its major climatic zones, seasonal patterns, key meteorological variables, regional variations, the monsoon system, extreme weather events, and observed climate change trends.

**General Guidelines**  
- Base all findings on primary sources, official data, and peer-reviewed literature. Preferred sources: Indian Meteorological Department (IMD), Ministry of Earth Sciences (MoES), World Bank Climate Data, IPCC reports, and India’s National Action Plan on Climate Change.
- If

/tmp/ipykernel_12205/2514999934.py:37: FutureWarning: Parameter 'structured_output' was not specified. Currently it defaults to False, but in version 1.25, the default will change to True. To suppress this warning, explicitly set structured_output=True (new behavior) or structured_output=False (legacy behavior). See documentation at https://huggingface.co/docs/smolagents/tutorials/tools#structured-output-and-output-schema-support for more details.
  with MCPClient({"url": MCP_URL, "transport": "streamable-http"}) as mcp_tools:


╭────────────────────────────────────────── New run - coordinator_agent ──────────────────────────────────────────╮
│                                                                                                                 │
│ You are the LEAD RESEARCH COORDINATOR AGENT.                                                                    │
│                                                                                                                 │
│ The user has asked:                                                                                             │
│ """research the climate in india"""                                                                             │
│                                                                                                                 │
│ A detailed research plan has already been created:                                                              │
│                                                                                                                 │
│ """I need a detailed, structured research report on the climate of India. Please follow the instructions below  │
│ to gather, analyze, and present the information. Do not complete the research yourself; rather, use these       │
│ instructions as the task specification.                                                                         │
│                                                                                                                 │
│ **Objective**                                                                                                   │
│ Produce a comprehensive analysis of India’s climate, covering its major climatic zones, seasonal patterns, key  │
│ meteorological variables, regional variations, the monsoon system, extreme weather events, and observed climate │
│ change trends.                                                                                                  │
│                                                                                                                 │
│ **General Guidelines**                                                                                          │
│ - Base all findings on primary sources, official data, and peer-reviewed literature. Preferred sources: Indian  │
│ Meteorological Department (IMD), Ministry of Earth Sciences (MoES), World Bank Climate Data, IPCC reports, and  │
│ India’s National Action Plan on Climate Change.                                                                 │
│ - If a specific time period is not prescribed, use the most recent 30‑year climatological normal (e.g.,         │
│ 1991‑2020) for averages. For trends, reference at least 50‑100 years of data where available.                   │
│ - When a dimension is left open (e.g., exactly which cities to include), use your judgment to provide a         │
│ representative and balanced picture. Flag such choices explicitly.                                              │
│ - Write in the first person from my perspective (e.g., “I want to understand…”), but deliver the final report   │
│ in an objective, third‑person narrative.                                                                        │
│                                                                                                                 │
│ **Report Structure & Required Content**                                                                         │
│                                                                                                                 │
│ 1. **Introduction**                                                                                             │
│    - Brief overview of India’s geographical and topographical influences on climate (latitudinal range,         │
│ Himalayas, coastline, Thar Desert).                                                                             │
│    - Statement on the dominant role of the monsoon.   

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'initialize_subagent' with arguments: {'subtask_description': "Conduct a top-down geographical    │
│ assessment of India's climate system by analyzing its latitudinal extent (from near the Tropics to the          │
│ Himalayan latitudes), the influence of the Himalayan range as a climatic barrier, the moderating effect of the  │
│ Indian Ocean coastline, and the arid conditions of the Thar Desert. Identify how these physical features shape  │
│ temperature gradients, precipitation zones, and seasonal transitions. Synthesize this into a concise,           │
│ first-person description that sets the stage for the monsoon’s dominance. Base this on official sources such as │
│ the Indian Meteorological Department (IMD), Ministry of Earth Sciences (MoES), and peer-reviewed geoscientific  │
│ literature (e.g., IPCC AR6, Climate Change 2023).", 'subtask_id': 'intro_geography', 'subtask_title':           │
│ "Geographical and Topographical Influences on India's Climate"}                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Step 4: Duration 3.83 seconds| Input tokens: 21,344 | Output tokens: 1,505]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error while generating output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/groq/openai/v1/chat/completions' 
(Request ID: Root=1-6a26fc83-7d8849f5485f944d7057c146;3c2db62a-bec7-4cb2-9fcb-f88364d5975a)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. 
Alternatively, subscribe to PRO to get 20x more included usage.

[Step 5: Duration 0.07 seconds]

AgentGenerationError: Error while generating output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/groq/openai/v1/chat/completions' (Request ID: Root=1-6a26fc83-7d8849f5485f944d7057c146;3c2db62a-bec7-4cb2-9fcb-f88364d5975a)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

In [47]:
print(result)


# Climate of India: A Comprehensive Analysis

## Executive Summary

India exhibits one of the world's most diverse climate systems, spanning tropical, subtropical, arid, semi-arid, temperate, alpine, and coastal climatic zones across approximately 3.29 million km². The country's climate is primarily governed by the South Asian Monsoon system, the Himalayas, surrounding oceans (Arabian Sea, Bay of Bengal, and Indian Ocean), and complex topography.

Key climate characteristics include mean annual temperatures ranging from below 0°C in high Himalayan regions to above 28°C in coastal and desert areas, annual precipitation from less than 100 mm in western Rajasthan to over 11,000 mm in parts of Meghalaya, and strong seasonal variability driven by monsoon circulation. Climate change signals are evident, with rising temperatures, increasing heatwave frequency, shifting rainfall patterns, and more intense extreme weather events.

## Geographic and Spatial Framework

### National Boundaries an